# Fako Online - Styled Scene Server (Kaggle)
**SD 1.5 + AnimateDiff Motion Adapter**

Generates images (SD 1.5) and styled character scenes (AnimateDiff).
Models loaded lazily on first API call to save VRAM.

### Instructions
1. Enable **GPU T4 x2**
2. Add datasets: `kingtechie/sd15-model`, `kingtechie/animatediff-model`
3. Run all cells
4. Server starts on port 8002

In [ ]:
# Cell 1: Setup paths + discover model structure
import os, glob

WORKING_DIR = "/kaggle/working/outputs"
os.makedirs(WORKING_DIR, exist_ok=True)

SD15_DIR = "/kaggle/input/sd15-model"
ANIMEDIFF_DIR = "/kaggle/input/animatediff-model"

# Check for nested subdirectories (like SadTalker issue)
for base_name in ["sd15-model", "animatediff-model"]:
    base_path = f"/kaggle/input/{base_name}"
    if os.path.exists(base_path):
        items = os.listdir(base_path)
        print(f"{base_name} contents: {items}")
        # Check if model_index.json is inside a subdirectory
        for item in items:
            sub = os.path.join(base_path, item)
            if os.path.isdir(sub) and os.path.exists(os.path.join(sub, "model_index.json")):
                print(f"  Found model_index.json in subdirectory: {item}")
            if os.path.isdir(sub) and os.path.exists(os.path.join(sub, "config.json")):
                print(f"  Found config.json in subdirectory: {item}")
    else:
        print(f"WARNING: {base_path} does not exist!")

# Check SD15 has required files
sd15_index = os.path.exists(os.path.join(SD15_DIR, "model_index.json"))
print(f"\nSD15 model_index.json exists: {sd15_index}")
print(f"SD15_DIR: {SD15_DIR}")
print(f"ANIMEDIFF_DIR: {ANIMEDIFF_DIR}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q diffusers transformers accelerate
!pip install -q fastapi uvicorn python-multipart
!pip install -q imageio[ffmpeg]
!pip install -q pyngrok
print("Dependencies installed!")

In [ ]:
# Cell 3: Import core libraries
import torch
import gc

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Lazy model loading — AnimateDiff + SD 1.5 loaded on first API call

_sd_pipe = None
_animatediff_pipe = None

def get_sd_pipe():
    global _sd_pipe
    if _sd_pipe is None:
        print("Loading SD 1.5 into GPU...")
        from diffusers import StableDiffusionPipeline
        _sd_pipe = StableDiffusionPipeline.from_pretrained(
            SD15_DIR,
            torch_dtype=torch.float16,
            safety_checker=None
        )
        _sd_pipe.to("cuda")
        print("SD 1.5 loaded!")
    return _sd_pipe

def get_animatediff_pipe():
    global _animatediff_pipe
    if _animatediff_pipe is None:
        print("Loading AnimateDiff pipeline into GPU...")
        from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

        # Load motion adapter from the animatediff dataset
        adapter_path = os.path.join(ANIMEDIFF_DIR, "motion_adapter")
        if not os.path.exists(adapter_path):
            # Try glob for nested structure
            adapter_files = glob.glob(f"{ANIMEDIFF_DIR}/**/pytorch_model.bin", recursive=True)
            if adapter_files:
                adapter_path = os.path.dirname(adapter_files[0])
                print(f"Found motion adapter at: {adapter_path}")

        adapter = MotionAdapter.from_pretrained(
            adapter_path,
            torch_dtype=torch.float16
        )

        # Load SD 1.5 as base, attach motion adapter
        _animatediff_pipe = AnimateDiffPipeline.from_pretrained(
            SD15_DIR,
            motion_adapter=adapter,
            torch_dtype=torch.float16,
            safety_checker=None
        )
        _animatediff_pipe.scheduler = DDIMScheduler.from_config(
            _animatediff_pipe.scheduler.config,
            beta_schedule="linear",
            clip_sample=False,
            timestep_spacing="linspace",
            steps_offset=1
        )
        _animatediff_pipe.to("cuda")
        print("AnimateDiff loaded!")
    return _animatediff_pipe

def unload_sd():
    global _sd_pipe
    if _sd_pipe is not None:
        del _sd_pipe
        _sd_pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        print("SD 1.5 unloaded from GPU")

def unload_animatediff():
    global _animatediff_pipe
    if _animatediff_pipe is not None:
        del _animatediff_pipe
        _animatediff_pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        print("AnimateDiff unloaded from GPU")

print("Lazy loading functions ready!")
print("Models load on first API call, unload after each use.")

In [ ]:
# Cell 5: Generation functions
from PIL import Image
from diffusers.utils import export_to_video

def generate_image(prompt, width=512, height=512):
    sd_pipe = get_sd_pipe()
    image = sd_pipe(
        prompt=prompt,
        width=width,
        height=height,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    output_path = f"{WORKING_DIR}/generated_image.png"
    image.save(output_path)
    # Move to CPU after use
    unload_sd()
    return output_path

def generate_styled_scene(image_path, prompt, style_prompt, duration=5, num_frames=16):
    ad_pipe = get_animatediff_pipe()
    init_image = Image.open(image_path).resize((512, 512))
    video_frames = ad_pipe(
        prompt=f"{prompt}, {style_prompt}",
        image=init_image,
        num_frames=num_frames,
        num_inference_steps=20,
        guidance_scale=7.5,
        height=512,
        width=512
    ).frames[0]
    output_path = f"{WORKING_DIR}/styled_scene.mp4"
    export_to_video(video_frames, output_path, fps=16)
    # Move to CPU after use
    unload_animatediff()
    return output_path

print("Generation functions defined!")

In [ ]:
# Cell 6: FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Fako Online - Styled Scene API")

@app.get("/health")
async def health():
    return {"status": "ok", "models": ["sd1.5", "animatediff"]}

@app.post("/generate-image")
async def api_generate_image(text_prompt: str = Form(...), width: int = Form(512), height: int = Form(512)):
    try:
        image_path = generate_image(text_prompt, width, height)
        return FileResponse(image_path, media_type="image/png")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-styled")
async def api_generate_styled(
    image: UploadFile = File(...),
    text_prompt: str = Form(...),
    style_prompt: str = Form(...),
    duration: int = Form(5)
):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        video_path = generate_styled_scene(image_path, text_prompt, style_prompt, duration)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

print("FastAPI server defined!")

In [ ]:
# Cell 7: Start server (threaded to avoid asyncio conflict with Jupyter)
import threading
import time
from pyngrok import ngrok

ngrok.set_auth_token("3JbM9BB0RTMJmMJtnx3EVCXlAoY_88KRTp3xpt6SGnX5ELFnS")

public_url = ngrok.connect(8002)
print(f"\n=== Server running ===")
print(f"Public URL: {public_url}", flush=True)
print(f"Use this URL in your local .env as KAGGLE_API_URL", flush=True)

with open("/kaggle/working/ngrok_url.txt", "w") as f:
    f.write(str(public_url))
print(f"URL saved to /kaggle/working/ngrok_url.txt", flush=True)

print(f"\nStarting FastAPI server on port 8002...", flush=True)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8002)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

for i in range(120):
    time.sleep(60)
print("Server stopped after 2 hours.")